# Sesión 5 — IA y análisis textual: escuchar lo que dice el corpus (Python)

Imaginate que tenés una caja con 22 recortes de prensa sobre un conflicto en el puerto. Si fueran 22, los leés y listo. Pero ¿y si fueran 2.200? ¿O 22.000? Ahí ya no alcanza el ojo humano. El **análisis textual** no reemplaza la lectura atenta: nos da una primera radiografía del corpus. Nos dice qué palabras pesan, dónde aparece un término que nos importa, cómo se reparte la cobertura. Es como pararse arriba de una loma antes de bajar a caminar el terreno.

El corpus es **sintético** (ficticio), inspirado en la prensa y la conflictividad obrera del Río de la Plata. Narra un conflicto portuario que va de la tensión a la asamblea, de ahí al paro y, finalmente, a la negociación y el acuerdo. Nuestra apuesta de hoy: que ese **relato del conflicto deje rastros en las frecuencias de palabras**.

Una aclaración de método que repito siempre: la IA nos escribe el código, pero el rigor lo ponemos nosotros. Todo lo que la máquina cuente, lo vamos a **validar a mano**. La IA se equivoca y alucina; quien investiga, valida.

## El prompt

Esto es lo que le pediríamos a la IA, en lenguaje natural, sin tecnicismos:

> "Tengo un CSV de notas de prensa con las columnas id, fecha, diario, seccion, titular y texto. Quiero cargarlo en Python con pandas, unir el titular con el texto, pasar todo a minúsculas, separar en palabras (tokenizar), sacar una lista corta de palabras vacías en español y mostrarme el top 15 de palabras más frecuentes. Usá pandas, el módulo re y collections.Counter. Comentá el código en español."

## El código que la IA nos devolvió

Cargamos las librerías y leemos el corpus.

In [ ]:
# pandas para manejar la tabla, re para cortar el texto en palabras,
# Counter para contar de forma rápida y clara
import pandas as pd
import re
from collections import Counter

# Leemos el CSV del corpus. La ruta es relativa a la carpeta de la sesión.
notas = pd.read_csv("../datos/notas_prensa.csv")

# Miramos las primeras filas para saber con qué trabajamos
notas.head()

Antes de contar nada, preparamos el texto: unimos titular y cuerpo, y bajamos todo a minúsculas. Esto es clave: para una computadora, "Paro" y "paro" son dos palabras distintas. Si no unificamos, contamos mal.

In [ ]:
# Unimos titular + texto en una sola columna y pasamos a minúsculas.
# .str.lower() unifica mayúsculas y minúsculas: "Paro" y "paro" cuentan igual.
notas["contenido"] = (notas["titular"] + " " + notas["texto"]).str.lower()

# Espiamos el primer documento ya preparado
notas["contenido"].iloc[0]

Ahora **tokenizamos**: cortamos cada nota en palabras sueltas. Usamos una expresión regular que se queda con secuencias de letras (incluyendo acentos y la ñ) y descarta números y signos de puntuación.

In [ ]:
# Esta función toma un texto y devuelve la lista de palabras.
# \b...\b y [a-záéíóúñü]+ se queda solo con letras (acentos y ñ incluidos).
def tokenizar(texto):
    return re.findall(r"[a-záéíóúñü]+", str(texto))

# Aplicamos la tokenización a cada nota y juntamos TODAS las palabras del corpus
todas_las_palabras = []
for contenido in notas["contenido"]:
    todas_las_palabras.extend(tokenizar(contenido))

# Cantidad total de palabras (tokens) en todo el corpus
len(todas_las_palabras)

Las palabras más comunes de cualquier texto en español son "de", "la", "el", "que"... No nos dicen nada del conflicto. Se llaman **palabras vacías** (*stopwords*). Armamos una lista CORTA y las sacamos.

In [ ]:
# Lista corta de palabras vacías en español.
# La armamos a mano y a propósito: queremos ver y entender qué descartamos.
vacias = {"de", "la", "el", "en", "y", "a", "los", "las", "que", "del",
          "un", "una", "con", "por", "se", "su", "para", "no", "mas",
          "o", "lo", "al", "le"}

# Nos quedamos solo con las palabras que NO están en la lista de vacías
palabras_limpias = [p for p in todas_las_palabras if p not in vacias]

len(palabras_limpias)

Y ahora sí: el **top 15** de palabras más frecuentes. `Counter.most_common()` hace el trabajo pesado.

In [ ]:
# Counter cuenta cada palabra; most_common(15) devuelve las 15 más frecuentes
conteo = Counter(palabras_limpias)
top15 = conteo.most_common(15)

# Lo pasamos a un DataFrame para verlo prolijo
top15_df = pd.DataFrame(top15, columns=["palabra", "frecuencia"])
top15_df

Un gráfico de barras lo hace más legible de un vistazo.

In [ ]:
import matplotlib.pyplot as plt

# Invertimos el orden para que la barra más larga quede arriba
plt.figure(figsize=(8, 5))
plt.barh(top15_df["palabra"][::-1], top15_df["frecuencia"][::-1], color="steelblue")
plt.title("Top 15 de palabras en el corpus del conflicto portuario")
plt.xlabel("Frecuencia")
plt.tight_layout()
plt.show()

## ¿Qué hace y cómo lo validamos?

Mirá el top 15: ahí están **paro, huelga, asamblea, trabajadores, sindicato, puerto, conflicto, acuerdo**. ¿Te das cuenta? El relato del conflicto —de la tensión al acuerdo— está escrito en las frecuencias. Las palabras que más pesan son, justamente, las protagonistas del proceso. Eso es lo lindo del análisis textual: **la estructura del relato aparece en los números**.

Pero no nos creamos nada porque sí. **Validamos a mano.** Tomemos una palabra del top, por ejemplo "paro", y contemos de otra forma para chequear que el número coincide.

In [ ]:
# Validación cruzada: el Counter ya tiene el número; lo comparamos contando
# 'paro' directamente sobre la lista de palabras limpias.
segun_counter = conteo["paro"]
segun_lista = palabras_limpias.count("paro")

print("'paro' según Counter:", segun_counter)
print("'paro' contando la lista:", segun_lista)
print("¿Coinciden?", segun_counter == segun_lista)

Si los dos números coinciden, vamos bien. Si no cierran, hay que parar y revisar: tal vez la tokenización separó algo raro, o una palabra vacía se nos coló. **Ese chequeo es nuestro trabajo, no el de la IA.**

## Buscar un término y ver dónde aparece

Las frecuencias nos dicen QUÉ pesa. Pero como humanistas queremos saber DÓNDE. Si "paro" es importante, ¿en qué notas aparece? Volver a la fuente es el corazón del oficio.

In [ ]:
# Buscamos 'paro' como palabra entera dentro de cada nota.
# \b marca límite de palabra: así 'paro' no matchea dentro de 'disparo'.
patron = re.compile(r"\bparo\b")

# Nos quedamos con las notas cuyo contenido contiene el término
mascara = notas["contenido"].apply(lambda t: bool(patron.search(str(t))))
notas_con_paro = notas.loc[mascara, ["id", "fecha", "titular"]]

notas_con_paro

Fijate que ahora podemos ir a leer esas notas concretas. El análisis textual no nos saca de la fuente: nos lleva de vuelta a ella, pero con una brújula.

## Contar notas por sección

Otra pregunta de investigación: ¿en qué secciones del diario vivió este conflicto? ¿Fue solo "Gremiales" o se derramó a "Locales", "Política", "Economía"? El reparto por sección nos habla de cómo la prensa encuadró el tema.

In [ ]:
# value_counts() cuenta cuántas notas hay por sección, de mayor a menor
notas_por_seccion = notas["seccion"].value_counts()
notas_por_seccion

In [ ]:
# Lo graficamos para verlo de un vistazo
plt.figure(figsize=(7, 4))
notas_por_seccion[::-1].plot(kind="barh", color="darkorange")
plt.title("Notas por sección")
plt.xlabel("Cantidad de notas")
plt.tight_layout()
plt.show()

Si el conflicto se concentra en "Gremiales" pero también aparece en otras secciones, eso cuenta una historia: el tema desbordó la página sindical y se volvió noticia general. Esa lectura la hacés vos; los números solo te la señalan.

## Para jugar

Probá pedirle estas variaciones a la IA y mirá qué cambia:

1. "En vez de 'paro', buscá la palabra 'acuerdo' y mostrame en qué notas aparece. ¿Aparece más al principio o al final del conflicto? Ordená por fecha." (Pista: el relato debería mostrar el acuerdo hacia el final.)

2. "Sumá tres palabras vacías más a la lista —por ejemplo 'sus', 'fue', 'ha'— y volvé a calcular el top 15. ¿Cambió mucho el ranking? ¿Por qué te parece que sí o que no?"